In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('/kaggle/input/labels-for-topic-modeling-output/LLM_comparison.csv')
df.head()

In [ ]:
df.columns

In [ ]:
topic_word_cols = [
    'gpt5_topic_words',
    'geminipro_topic_words',
    'perplexity_topic_words',
    'grok_topic_words',
    'deepseek_topic_words',
    'claude_topic_words'
]

df['topic_words_consistent'] = (
    df[topic_word_cols]
    .astype(str) # Convert to string to ensure consistent comparison (especially if there are mixed types/NaT)
    .apply(lambda row: row.nunique(), axis=1) == 1
)
print(len(df['topic_words_consistent']), " consistent rows")

In [ ]:
pip install -U "transformers==4.36.2" "huggingface_hub==0.20.2" "sentence-transformers==2.6.1" "accelerate==0.25.0"

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("allenai-specter")

In [ ]:
llm_label_cols = ['gpt5_label', 'geminipro_label', 'perplexity_label', 'grok_label', 'deepseek_label', 'claude_label']
reference_col = 'gpt5_topic_words'

In [ ]:
print("Encoding text columns...")

df_embeddings = {}
df_embeddings['gpt5_topic_words'] = model.encode(df[reference_col].astype(str).tolist(), show_progress_bar=True)

for col in llm_label_cols:
    df_embeddings[col] = model.encode(df[col].astype(str).tolist(), show_progress_bar=True)

# --- Compute cosine similarities ---
for col in llm_label_cols:
    similarities = cosine_similarity(df_embeddings[col], df_embeddings['gpt5_topic_words'])
    df[f'similarity_{col}'] = [similarities[i, i] for i in range(len(df))]  # diagonal: same topic row vs gpt5 topic

# --- Summary Statistics ---
similarity_cols = [f'similarity_{col}' for col in llm_label_cols]
summary = df[similarity_cols].describe().T
summary['mean_similarity'] = summary['mean']
summary['median_similarity'] = df[similarity_cols].median().values
summary = summary[['mean_similarity', 'median_similarity', 'std', 'min', 'max']]

print("\n--- Similarity Summary ---")
print(summary)

# --- Visualization ---
plt.figure(figsize=(10, 6))
sns.boxplot(data=df[similarity_cols])
plt.title("Distribution of Cosine Similarities with GPT-5 Topics")
plt.ylabel("Cosine Similarity")
plt.xticks(rotation=30)
plt.show()

plt.figure(figsize=(10, 6))
sns.heatmap(df[similarity_cols].corr(), annot=True, cmap="Blues", fmt=".2f")
plt.title("Correlation Between LLM Topic-Label Similarities")
plt.show()

# --- Additional Insights ---
insights = {}

for col in similarity_cols:
    mean_sim = df[col].mean()
    if mean_sim > 0.8:
        insights[col] = f"High conceptual alignment with GPT-5 topics (mean={mean_sim:.2f})."
    elif mean_sim > 0.6:
        insights[col] = f"Moderate semantic overlap with GPT-5 topics (mean={mean_sim:.2f})."
    else:
        insights[col] = f"Low similarity — likely distinct topic interpretations (mean={mean_sim:.2f})."

print("\n--- Insights ---")
for k, v in insights.items():
    print(f"{k}: {v}")

# --- Optional: Rank models by similarity ---
ranking = summary['mean_similarity'].sort_values(ascending=False)
print("\n--- LLMs Ranked by Semantic Alignment to GPT-5 Topics ---")
print(ranking)